# Checkpoint 11: prepare a fair comparison

The earlier notebooks exposed two problems: some trends are missing, and high accuracy can hide poor incident detection. We now prepare the data so candidate models face the same test.

Use older shipments for training, later shipments to compare models, and the final group for evaluation. A shipment stays in one group. The split uses first decision times and approximate 60/20/20 boundaries; labels do not determine those boundaries.

Training labels must be available by validation start, and validation labels by test start. Both classes must finish the six-hour outcome window and the 48-hour reporting allowance before entering a partition.

For the final evaluation, the latest supplied decision timestamp is our assumed observation cutoff. It does not prove that auditing is complete. We exclude windows that are too recent and keep the mature-record completeness assumption explicit.

`personal/notebooks/support/workflow.py` prepares these experiments. Its feature calculations now use the package functions shared with the engine. Run this notebook from the top; no previous kernel state is needed.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/dispatch_risk/contracts.py').is_file())
# Import the repository source even when editable-install paths are unavailable.
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'personal' / 'notebooks' / 'support'))
import workflow as wf
import pandas as pd
import numpy as np
from IPython.display import display
train, validation, raw_features, reports, manifest = wf.development_data()
display(pd.DataFrame([{"group": "train", "rows": len(train), "shipments": train.shipment_id.nunique(), "positives": int(train.label.sum())},
                      {"group": "validation", "rows": len(validation), "shipments": validation.shipment_id.nunique(), "positives": int(validation.label.sum())},
                      {"group": "test (labels not examined)", "rows": int(raw_features.cohort.eq("test").sum()), "shipments": raw_features.loc[raw_features.cohort.eq("test"),"shipment_id"].nunique(), "positives": None}]))
print(manifest)


                        group  rows  shipments  positives
0                       train  1026        343       77.0
1                  validation   306        103       21.0
2  test (labels not examined)   360        120        NaN
{'validation_start': '2026-02-15T08:00:00+00:00', 'test_start': '2026-03-02T08:00:00+00:00', 'observation_cutoff': '2026-03-17T11:00:00+00:00', 'grace_hours': 48, 'lookback_hours': 3, 'cohort_rule': '60/20/20 by shipment first decision time; ties assigned by time', 'completeness_assumption': 'Reports complete for horizons ending >=48h before the specified label cutoff; not guaranteed by input', 'source_sha256': {'events.jsonl': '2ce178c72e6d4c2e95c46eeff010abe677dd667fb58881612ad414ad24afad98', 'decision_times.jsonl': '6b101d088add123ef2242efb02eb0574f7a52dddbb918ca927c051fec70d45e8', 'labels.jsonl': '351bad0c44975e00c6db7b7b5701a3d53739b78c4c3318a4174d2895186a06f0'}}


## 1. Check chronology and exclusions

A shared incident can label multiple checkpoints within a shipment, so rows are correlated. Shipment-separated evaluation reduces cross-group dependence, while later uncertainty estimates should resample shipments rather than pretend all rows are independent.

Maturity creates a gap between groups' used decision times. Earlier training horizons close before validation begins; validation horizons and their reporting allowances end before the final test starts. No model selection can use test outcomes.


In [2]:
assert set(train.shipment_id).isdisjoint(validation.shipment_id)
test_ids = set(raw_features.loc[raw_features.cohort.eq("test"),"shipment_id"])
assert test_ids.isdisjoint(train.shipment_id) and test_ids.isdisjoint(validation.shipment_id)
assert (train.decision_time + wf.HORIZON + pd.Timedelta(hours=48) <= pd.Timestamp(manifest["validation_start"])).all()
assert (validation.decision_time + wf.HORIZON + pd.Timedelta(hours=48) <= pd.Timestamp(manifest["test_start"])).all()
assert train.decision_time.max() < validation.decision_time.min()
exclusions = pd.DataFrame([{"group": name, "cohort_rows": int(raw_features.cohort.eq(name).sum()), "eligible_rows": len(frame), "excluded_immature_rows": int(raw_features.cohort.eq(name).sum())-len(frame)} for name, frame in [("train",train),("validation",validation)]])
display(exclusions)
print("Chronology, shipment separation, and outcome maturity checks passed.")


        group  cohort_rows  eligible_rows  excluded_immature_rows
0       train         1080           1026                      54
1  validation          360            306                      54
Chronology, shipment separation, and outcome maturity checks passed.


## 2. Missing features and scaling

Median imputation replaces a missing numeric input with the middle training value. A separate missing flag records that it was not observed. If a feature is entirely empty during fitting, the library retains its column using a zero fallback; the missing flag distinguishes that fallback from a measurement. Indicators are present for every feature, including missingness first seen later.

Logistic regression will also standardize numeric values using the training mean and standard deviation after imputation. Tree candidates do not require scaling. Both methods keep original raw data unchanged. Every pipeline is freshly fitted using training rows only; calling transform on later rows must not change learned statistics.

Constant/redundant candidates remain a limitation, not an invitation to select features using test outcomes. The small predefined feature set is held fixed for this comparison.


In [3]:
preprocessor = wf.imputation_pipeline(scale=True)
X_train = preprocessor.fit_transform(train[wf.FEATURES])
imputer = preprocessor.named_transformers_["values"].named_steps["impute"]
stats_before = imputer.statistics_.copy()
X_validation = preprocessor.transform(validation[wf.FEATURES])
assert np.array_equal(stats_before, imputer.statistics_, equal_nan=True)
assert np.isfinite(X_train).all() and np.isfinite(X_validation).all()
display(pd.DataFrame({"feature":wf.FEATURES, "train_missing":train[wf.FEATURES].isna().sum().to_numpy(), "train_median": imputer.statistics_}))
print("Transformed dimensions:",X_train.shape,X_validation.shape)
output = ROOT / 'personal' / 'data' / 'tables'
output.mkdir(exist_ok=True,parents=True)
train.to_csv(output / "checkpoint11_train.csv",index=False)
validation.to_csv(output / "checkpoint11_validation.csv",index=False)
wf.save_json(ROOT / 'personal' / 'outputs' / 'notebook_results' / "split_policy.json",manifest)
print("Saved development tables and split policy; test outcomes were not summarized.")


                        feature  train_missing  train_median
0          latest_temperature_c              0        4.7200
1       measurement_age_minutes              0       60.0000
2         arrival_delay_minutes              0        2.0000
3           temperature_missing              0        0.0000
4             temperature_count              0        2.0000
5            temperature_mean_c             15        4.6695
6             temperature_max_c             15        4.9590
7  temperature_trend_c_per_hour            264        0.1850
8        temperature_span_hours             15        1.0000
Transformed dimensions: (1026, 18) (306, 18)
Saved development tables and split policy; test outcomes were not summarized.


## 3. What notebook 12 will do with these outputs

The split checks must pass and training/validation must contain both classes before comparing models. Notebook 12 uses this same split and feature schema to fit the constant baseline, logistic regression, a shallow tree, random forest, and gradient boosting. Each gets a fresh preprocessing pipeline. A deterministic, predefined validation selection rule chooses a candidate before any test evaluation.

This is one small synthetic temporal evaluation, not enough to justify deployment. No data balancing, calibration model, or alert threshold has been tuned.

Reference: [scikit-learn leakage and pipeline guidance](https://scikit-learn.org/stable/common_pitfalls.html).

**Try explaining this:** why must validation data use the training median instead of supplying its own replacement values?
